# Usuwanie zakłóceń impulsowych

**A:** z zakwestionowaną próbką. **B:** z ostatnią zaakceptowaną próbką.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from IPython.display import Audio, display


### Wczytanie nagrania


In [ ]:
dane, fs = sf.read("nagrania/02.wav", always_2d=True)
x = dane[:, 0]
print(f"Próbkowanie: {fs} Hz, czas: {len(x) / fs:.2f} s")


### Przebieg i odsłuch


In [ ]:
t = np.arange(len(x)) / fs
start, stop = int(0.3 * fs), int(0.5 * fs)
fig, ax = plt.subplots(2, 1, figsize=(11, 5), constrained_layout=True)
ax[0].plot(t, x, lw=0.5)
ax[0].set(title="Całe nagranie - kanał 1", xlabel="Czas [s]")
ax[1].plot(t[start:stop] * 1000, x[start:stop], ".-", lw=0.8, markersize=3)
ax[1].set(title="Fragment 300-500 ms - pierwszy trzask", xlabel="Czas [ms]")
for a in ax:
    a.set_ylabel("Amplituda")
    a.grid(alpha=0.3)
plt.show()

display(Audio(x, rate=fs, normalize=False))


## AR(4) i RLS

$$\varphi(k)=[x(k-1),x(k-2),x(k-3),x(k-4)]^T,$$

$$\hat x(k)=a^T(k-1)\varphi(k),\qquad e(k)=x(k)-\hat x(k).$$

$$J_k(a)=\sum_{i=4}^{k}\lambda^{k-i}[x(i)-a^T\varphi(i)]^2.$$

$$g=\frac{P\varphi}{\lambda+\varphi^TP\varphi},\qquad a\leftarrow a+g e,$$

$$P\leftarrow\frac{P-g\varphi^TP}{\lambda}.$$

Start: $a=0$, $P=1000I$. Błąd przed aktualizacją. Dla zakwestionowanej próbki: $a$ bez zmian, $P\leftarrow P/\lambda$.


In [ ]:
r = 4
lam = 0.999
okno = 221  # około 10 ms

def detekcja(x, podmien=False):
    a = np.zeros(r)
    P = 1000 * np.eye(r)
    z = x.copy()
    e = np.full(len(x), np.nan)
    prog = np.full(len(x), np.nan)
    maska = np.zeros(len(x), dtype=bool)
    a_hist = np.full((len(x), r), np.nan)

    for k in range(r, len(x)):
        phi = z[k-r:k][::-1]
        e[k] = x[k] - a @ phi
        if k >= r + okno:
            prog[k] = 3 * np.std(e[k-okno:k])
            maska[k] = abs(e[k]) > prog[k]

        if not maska[k]:
            q = P @ phi
            g = q / (lam + phi @ q)
            a = a + g * e[k]
            P = (P - np.outer(g, phi @ P)) / lam
        else:
            if podmien:
                z[k] = z[k-1]
            P = P / lam  # samo zapominanie, bez uczenia z próbki
        a_hist[k] = a
    return e, prog, maska, a_hist

e, prog, maska, a_hist = detekcja(x)


### A - parametry i błąd
Wykresy: co dziesiąta próbka.


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True, constrained_layout=True)
ax[0].plot(t[::10], a_hist[::10], lw=0.7)
ax[0].legend(["a1", "a2", "a3", "a4"], ncol=4)
ax[0].set(title="A - z zakwestionowaną próbką: parametry AR(4)", ylabel="Wartość parametru")
ax[1].plot(t[::10], e[::10], lw=0.5)
ax[1].set(title="A - z zakwestionowaną próbką: błąd predykcji", xlabel="Czas [s]", ylabel="Amplituda")
for a_plot in ax:
    a_plot.grid(alpha=0.3)
plt.show()


`e` - błędy, `a_hist` - parametry, `maska` - detekcje wariantu A.


## Detekcja - wariant A
$$|e(k)|>3\sigma(k).$$
$\sigma$: odchylenie poprzednich 221 błędów (około 10 ms), także zakwestionowanych. Pierwsze 221 predykcji: rozruch bez detekcji.


In [ ]:
print(f"Wariant A - podejrzane próbki: {maska.sum()} ({100 * maska.mean():.2f}%)")


In [ ]:
plt.figure(figsize=(11, 3))
plt.plot(t, np.abs(e), lw=0.5, label="|e|")
plt.plot(t, prog, lw=0.8, label="Próg 3σ")
plt.scatter(t[maska], np.abs(e[maska]), s=8, color="red", label="Detekcje")
plt.title("A - z zakwestionowaną próbką: detekcja")
plt.xlabel("Czas [s]")
plt.ylabel("Amplituda")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Interpolacja - wariant A
Naprawa serii do 4 próbek. Dla jednej próbki:
$$y(k)=\frac{x(k-1)+x(k+1)}{2}.$$


In [ ]:
def interpolacja(x, maska):
    granice = np.diff(np.r_[False, maska, False].astype(int))
    poczatki = np.flatnonzero(granice == 1)
    konce = np.flatnonzero(granice == -1)  # koniec poza serią

    y = x.copy()
    naprawione = np.zeros(len(x), dtype=bool)
    for start, stop in zip(poczatki, konce):
        if 0 < start and stop < len(x) and stop - start <= 4:
            y[start:stop] = np.linspace(x[start-1], x[stop], stop-start+2)[1:-1]
            naprawione[start:stop] = True

    return y, naprawione

y, naprawione = interpolacja(x, maska)
print(f"Wariant A - próbki objęte rekonstrukcją: {naprawione.sum()}")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3), sharex=True, constrained_layout=True)
ax[0].plot(t, y, lw=0.5)
ax[0].set(title="A - Sygnał po naprawie", ylabel="Amplituda")
ax[1].plot(t, x - y, lw=0.5)
ax[1].set(title="A - Różnica: wejście − wynik", ylabel="Zmiana amplitudy")
for a_plot in ax:
    a_plot.set_xlabel("Czas [s]")
    a_plot.grid(alpha=0.3)
plt.show()

n = int(0.1 * fs)
fig, ax = plt.subplots(1, 2, figsize=(12, 3), sharex=True, constrained_layout=True)
ax[0].plot(t[:n] * 1000, y[:n], lw=0.7)
ax[0].set(title="A - Po naprawie - pierwsze 100 ms", ylabel="Amplituda")
ax[1].plot(t[:n] * 1000, (x - y)[:n], lw=0.7)
ax[1].set(title="A - Różnica - pierwsze 100 ms", ylabel="Zmiana amplitudy")
for a_plot in ax:
    a_plot.set_xlabel("Czas [ms]")
    a_plot.grid(alpha=0.3)
plt.show()



## Porównanie A/B
**A:** $z(k)=x(k)$. **B:** po detekcji $z(k)=z(k-1)$.

$$\varphi(k)=[z(k-1),z(k-2),z(k-3),z(k-4)]^T.$$

W obu: brak uczenia na zakwestionowanej próbce; próg ze wszystkich błędów; końcowa naprawa interpolacją. Powtarzanie wartości w B dotyczy tylko historii modelu.


In [ ]:
e_b, prog_b, maska_b, a_hist_b = detekcja(x, podmien=True)
y_b, naprawione_b = interpolacja(x, maska_b)

warianty = {
    "A - z zakwestionowaną próbką": (y, maska, naprawione),
    "B - z ostatnią zaakceptowaną próbką": (y_b, maska_b, naprawione_b),
}


### Statystyki
Naprawione: próbki objęte interpolacją. Pominięte serie: za długie lub na brzegu. RMSE zmiany mierzy ingerencję, nie jakość odszumiania.


In [ ]:
from IPython.display import Markdown

wiersze = ["| Wariant | Detekcje | Naprawione | Udział [%] | Pominięte serie | RMSE zmiany |",
           "|---|---:|---:|---:|---:|---:|"]
for nazwa, (wynik, detekcje, naprawy) in warianty.items():
    pominiete = detekcje & ~naprawy
    serie = np.count_nonzero(np.diff(np.r_[False, pominiete].astype(int)) == 1)
    rmse = np.sqrt(np.mean((wynik - x)**2))
    wiersze.append(f"| {nazwa} | {detekcje.sum()} | {naprawy.sum()} | "
                   f"{100 * naprawy.mean():.3f} | {serie} | {rmse:.6f} |")
display(Markdown("\n".join(wiersze)))


### Wykresy A/B
Całe nagranie i pierwsze 100 ms. Wspólne skale obu wariantów.


In [ ]:
for n, skala, jednostka in [(len(x), 1, "s"), (int(0.1 * fs), 1000, "ms")]:
    fig, ax = plt.subplots(2, 2, figsize=(12, 5), sharex=True,
                           sharey="col", constrained_layout=True)
    for i, (nazwa, (wynik, _, _)) in enumerate(warianty.items()):
        ax[i, 0].plot(t[:n] * skala, wynik[:n], lw=0.6)
        ax[i, 0].set(title=f"{nazwa}: po naprawie", ylabel="Amplituda")
        ax[i, 1].plot(t[:n] * skala, (x - wynik)[:n], lw=0.6)
        ax[i, 1].set(title=f"{nazwa}: wejście − wynik", ylabel="Różnica")
    for a_plot in ax.flat:
        a_plot.set_xlabel(f"Czas [{jednostka}]")
        a_plot.grid(alpha=0.3)
    plt.show()


### Odsłuch


In [ ]:
print("Wejście:")
display(Audio(x, rate=fs, normalize=False))
for nazwa, (wynik, _, _) in warianty.items():
    print(nazwa)
    display(Audio(wynik, rate=fs, normalize=False))
